# RF-DETR Training and Evaluation

This notebook contains the training, evaluation, and benchmarking
workflow for the RF-DETR model used in the research study.

The original experimental workflow is preserved to support
reproducibility and comparison with the YOLO-based models.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Stage 0 — Environment Setup

This stage prepares and verifies the environment required to run
the RF-DETR experiment.

### Tasks

- Install required dependencies
- Verify RF-DETR version
- Import required libraries

In [ ]:
# Verify the RF-DETR version used by the experiment.
# Keep the version consistent when reproducing the reported results.
!pip install "rfdetr[train,loggers]==1.8.1" PyYAML


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!rm -rf /content/sample_data

Imports the libraries
required for training and
evaluating same as the YOLO model family.




In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import shutil, subprocess
from google.colab import output
import sys
from pathlib import Path
import logging
import sys
import time
import yaml

In [ ]:
# Install the required packages for the experiment.
def git_update(repo_url="https://github.com/tommyngx/DentalYOLO.git"):
    %cd /content
    repo_name = Path(urlparse(repo_url).path).stem
    target = Path.cwd() / repo_name
    if target.exists():
        shutil.rmtree(target)  # remove
    subprocess.run(["git", "clone", repo_url, str(target)], check=True)
    return target
git_update("https://github.com/tommyngx/DentalYOLO.git")

/content


PosixPath('/content/DentalYOLO')

In [ ]:
%cd /content/DentalYOLO
%pip install -qr requirements.txt  # install

/content/DentalYOLO
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rfdetr 1.8.3 req

In [ ]:
# Verify the library versions used by the experiment.
# Keep these versions consistent when reproducing the reported results.
CUSTOM_ROOT = Path("/content/DentalYOLO/ultralytics")
for m in list(sys.modules):
    if m.startswith("ultralytics"):
        del sys.modules[m]
sys.path.insert(0, str(CUSTOM_ROOT))
import ultralytics
from ultralytics import YOLO
print("Using ultralytics from:", ultralytics.__file__)
logger = logging.getLogger("ultralytics")
if len(logger.handlers) > 1:
    logger.handlers = logger.handlers[:1]
logger.propagate = False
print("Number of ultralytics handlers after fix:", len(logger.handlers))
print(ultralytics.__version__)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using ultralytics from: /content/DentalYOLO/ultralytics/__init__.py
Number of ultralytics handlers after fix: 1
8.4.75


In [ ]:
from pathlib import Path
from ultralytics import YOLO
import time
T_START = time.time()


## Stage 1 — Configuration

This stage defines the dataset, model, training, evaluation, and
output configuration used throughout the notebook.

### Task — Experiment Configuration

Update the dataset, model, or experiment parameters here when running
a different experiment.

> **Important:** Parameters used to reproduce the reported research
> results should not be changed unless a new experiment is intended.

In [ ]:
import os
from dataclasses import dataclass


In [ ]:
@dataclass
class Config:
    # PROJECT
    project_dir: str = "/content/drive/MyDrive/Your-Project"
    experiment_name: str = "experiment_name"

    # DATA
    # Change this to the dataset you want to use.
    dataset_name: str = "dataset_name"
    # Change this to your dataset ZIP file.
    zip_dataset: str = os.path.join(
        project_dir,
        "datasets",
        f"{dataset_name}.zip"
    )

    # MODEL
    # Change this to select the model used in the experiment.
    model_name: str = "rfdetr-medium-181"

    # TRAINING
    # Only epochs is set explicitly.
    # All other training settings come from RF-DETR defaults.
    epochs: int = 100

cfg = Config()

# PATH SETUP
cfg.output_root = os.path.join(cfg.project_dir, "output")

cfg.output_dir = os.path.join(
    cfg.output_root,
    cfg.dataset_name,
    cfg.model_name,
)
cfg.result_dir = os.path.join(
    cfg.output_dir,
    cfg.experiment_name,
)
# This file defines the dataset paths, number of classes, and class names.
cfg.data_dir = os.path.join("/content", cfg.dataset_name)
cfg.yaml_path = os.path.join(cfg.data_dir, "data.yaml")
# create dirs
os.makedirs(cfg.output_dir, exist_ok=True)

print("Output dir:", cfg.output_dir)
print("Data dir:", cfg.data_dir)


Output dir: /content/drive/MyDrive/Oral_Disease_Detection/output/dental_opg_2024/rfdetr-medium
Data dir: /content/dental_opg_2024
Coco dir: /content/dental_opg_2024_coco


In [ ]:
# Use the framework's built-in default seed and determinism settings.
# No manual seeding is applied here.
print("Training will use each framework's default runtime settings (only epochs=100 is explicit).")


Training will use each framework's default runtime settings (only epochs=100 is explicit).


## Stage 2 — Dataset Preparation

This stage prepares and verifies the dataset before model training.

### Tasks

- Locate the dataset
- Read the dataset configuration
- Verify image and label directories
- Check dataset structure
- Prepare the dataset for training and evaluation

In [ ]:
# Extract the dataset ZIP archive if it has not already been prepared.
!unzip -q {cfg.zip_dataset} -d /content/
print("Dataset ready:", cfg.data_dir)

Dataset ready: /content/dental_opg_2024


In [ ]:
# Verify that the expected image and label directories are available.
assert os.path.exists(cfg.yaml_path), "Missing data.yaml"
assert os.path.exists(os.path.join(cfg.data_dir, "train")), "Missing train folder"
assert os.path.exists(os.path.join(cfg.data_dir, "val")), "Missing val folder"

print("Dataset structure OK ✅")

In [ ]:
from ultralytics import YOLO
from ultralytics import settings

settings.update({"datasets_dir": ""})

In [ ]:
import yaml

def fix_data_yaml(yaml_path, dataset_name):
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    # Update path
    new_path = f"/content/{dataset_name}"
    data["path"] = new_path

    with open(yaml_path, "w") as f:
        yaml.safe_dump(data, f, sort_keys=False)

    print(f"Updated path -> {new_path}")
    return data

fix_data_yaml(cfg.yaml_path, cfg.dataset_name)

Updated path -> /content/dental_opg_2024


{'path': '/content/dental_opg_2024',
 'train': 'train/images',
 'val': 'val/images',
 'test': 'test/images',
 'nc': 6,
 'names': {0: 'Healthy teeth',
  1: 'Infection',
  2: 'Impacted teeth',
  3: 'Broken-down crown or root (BDC/BDR)',
  4: 'Fractured teeth',
  5: 'Caries'}}

### Format Dataset - COCO

In [ ]:
!pip install yolococo


In [ ]:
import json
import yaml
from pathlib import Path
from yolococo import yolo_to_coco

# load class names
def load_class_names(yaml_path):
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    names = data.get("names", {})

    if isinstance(names, dict):
        class_list = [names[i] for i in sorted(names.keys())]
    else:
        class_list = names

    return class_list

# save classes.txt
def create_classes_txt(class_names, save_path):
    with open(save_path, "w") as f:
        for name in class_names:
            f.write(name + "\n")

class_names = load_class_names(cfg.yaml_path)

classes_txt_path = Path(cfg.coco_dir) / "classes.txt"
create_classes_txt(class_names, classes_txt_path)

# convert
def convert_split(split):
    split_map = {
        "train": "train",
        "val": "valid",
        "test": "test"
    }

    target_split = split_map[split]

    images_dir = Path(cfg.data_dir) / split / "images"
    labels_dir = Path(cfg.data_dir) / split / "labels"

    output_dir = Path(cfg.coco_dir) / target_split
    output_dir.mkdir(parents=True, exist_ok=True)

    coco = yolo_to_coco(
        images_dir=images_dir,
        labels_dir=labels_dir,
        classes_path=classes_txt_path,
        image_size=None,
        info={"description": f"{cfg.dataset_name} {target_split}"},
        supercategory="object"
    )

    ann_path = output_dir / "_annotations.coco.json"
    with open(ann_path, "w", encoding="utf-8") as f:
        json.dump(coco, f, indent=2)

    # copy images
    os.system(f"cp -r {images_dir} {output_dir}")

    print(f"✅ Converted {split} → {target_split}")
for split in ["train", "val", "test"]:
    convert_split(split)

YOLO→COCO:   0%|          | 0/558 [00:00<?, ?img/s]

✅ Converted train → train


YOLO→COCO:   0%|          | 0/23 [00:00<?, ?img/s]

✅ Converted val → valid


YOLO→COCO:   0%|          | 0/23 [00:00<?, ?img/s]

✅ Converted test → test


In [ ]:
import json
from pathlib import Path

def fix_coco_paths(split):
    ann_path = Path(cfg.coco_dir) / split / "_annotations.coco.json"

    with open(ann_path, "r") as f:
        data = json.load(f)

    for img in data["images"]:
        img["file_name"] = f"images/{img['file_name']}"  # 🔥 FIX

    with open(ann_path, "w") as f:
        json.dump(data, f, indent=2)

    print(f"✅ Fixed paths for {split}")

# apply
for split in ["train", "valid", "test"]:
    fix_coco_paths(split)

✅ Fixed paths for train
✅ Fixed paths for valid
✅ Fixed paths for test


In [ ]:
import json

ann_path = os.path.join(cfg.coco_dir, "train", "_annotations.coco.json")

with open(ann_path) as f:
    data = json.load(f)

print("Images:", len(data["images"]))
print("Annotations:", len(data["annotations"]))
print("Categories:", data["categories"])

Images: 558
Annotations: 5642
Categories: [{'id': 0, 'name': 'Healthy teeth', 'supercategory': 'object'}, {'id': 1, 'name': 'Infection', 'supercategory': 'object'}, {'id': 2, 'name': 'Impacted teeth', 'supercategory': 'object'}, {'id': 3, 'name': 'Broken-down crown or root (BDC/BDR)', 'supercategory': 'object'}, {'id': 4, 'name': 'Fractured teeth', 'supercategory': 'object'}, {'id': 5, 'name': 'Caries', 'supercategory': 'object'}]


In [ ]:
import json
import os

def get_num_classes_from_coco(coco_path):
    with open(coco_path, "r") as f:
        coco = json.load(f)

    categories = coco.get("categories", [])
    num_classes = len(categories)

    print("Detected num_classes:", num_classes)
    print("Categories:", [c["name"] for c in categories])

    return num_classes

In [ ]:
coco_path = os.path.join(cfg.coco_dir, "test/_annotations.coco.json")
num_classes = get_num_classes_from_coco(coco_path)

Detected num_classes: 10
Categories: ['Broken_Root', 'PCT', 'Free_R_Max', 'Free_L_Max', 'Not_Free_Max', 'Not_Free_Center_Max', 'Free_R_Mand', 'Free_L_Mand', 'Not_Free_Mand', 'Not_Free_Center_Mand']


## Stage 3 — Model Training

This stage trains RF-DETR using the configured dataset and
experimental settings.

### Tasks

- Configure the trainer
- Start training
- Save checkpoints
- Save training logs

In [ ]:
from rfdetr import RFDETRMedium

def train_rfdetr(cfg):
    # Load the RF-DETR model configuration.
    model = RFDETRMedium(num_classes=num_classes)
    model.train(
        dataset_dir=cfg.coco_dir,
        epochs=cfg.epochs,
        output_dir=cfg.result_dir,
    )

    print("Saved run to:", cfg.result_dir)
    return model

model = train_rfdetr(cfg)


Val (Epoch 96/100) — Overall Metrics              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃           mAP            ┃  mAR   ┃         F1 sweep         ┃
┡━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━╇━━━━━━━━┯━━━━━━━━┯━━━━━━━━┩
│ 50:95  │   50   │   75   │  @500  │   F1   │  Prec  │ Recall │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 0.4692 │ 0.7459 │ 0.5260 │ 0.6747 │ 0.7249 │ 0.7020 │ 0.7631 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┘
                  Val (Epoch 96/100) — Per-class Metrics                  
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Class                ┃ AP 50:95 ┃     AR ┃     F1 ┃ Precision ┃ Recall ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ Broken_Root          │   0.2829 │ 0.5083 │ 0.5974 │    0.4894 │ 0.7667 │
│ PCT                  │   0.0672 │ 0.4808 │ 0.2642 │    0.2593 │ 0.2692 │
│ Free_R_Max           │   0.5455 │ 0.7333 │ 0.8333 │    0.8333 │ 0.8333 │
│ Free_L_Max           │   0.4010 │ 0.6429 │ 0.5000 │    0.6000 │ 0.4286 │
│ Not_Free_Max         │   0.7026 │ 0.7583 │ 0.9000 │    0.8182 │ 1.0000 │
│ Not_Free_Center_Max  │   0.5822 │ 0.7400 │ 0.9091 │    0.8333 │ 1.0000 │
│ Free_R_Mand          │   0.5870 │ 0.6500 │ 1.0000 │    1.0000 │ 1.0000 │
│ Free_L_Mand          │   0.5733 │ 0.6333 │ 0.8696 │    0.9091 │ 0.8333 │
│ Not_Free_Mand        │   0.6782 │ 0.7500 │ 0.8750 │    0.7778 │ 1.0000 │
│ Not_Free_Center_Mand │   0.2722 │ 0.8500 │ 0.5000 │    0.5000 │ 0.5000 │
└──────────────────────┴──────────┴────────┴────────┴───────────┴────────┘

In [ ]:
import pandas as pd

history_path = os.path.join(cfg.result_dir, "metrics.csv")
df = pd.read_csv(history_path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


# Clean val (drop NaN)
val_df = df.dropna(subset=[
    "val/loss",
    "val/mAP_50",
    "val/mAP_50_95",
    "val/precision",
    "val/recall"
])

# Clean train
train_df = df.dropna(subset=["train/loss"])

# Merge theo epoch
merged = pd.merge(
    train_df[["epoch", "train/loss"]],
    val_df[[
        "epoch",
        "val/loss",
        "val/mAP_50",
        "val/mAP_50_95",
        "val/precision",
        "val/recall"
    ]],
    on="epoch",
    how="inner"
)

# Plot
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Training vs Validation Metrics", fontsize=16)

# Loss
axes[0,0].plot(merged["epoch"], merged["train/loss"], label="Train Loss")
axes[0,0].plot(merged["epoch"], merged["val/loss"], label="Val Loss")
axes[0,0].set_title("Loss")
axes[0,0].legend()

# mAP50
axes[0,1].plot(merged["epoch"], merged["val/mAP_50"], label="mAP@50")
axes[0,1].set_title("mAP@50")

# mAP50-95
axes[0,2].plot(merged["epoch"], merged["val/mAP_50_95"], label="mAP@50-95")
axes[0,2].set_title("mAP@50-95")

# Precision
axes[1,0].plot(merged["epoch"], merged["val/precision"], label="Precision")
axes[1,0].set_title("Precision")

# Recall
axes[1,1].plot(merged["epoch"], merged["val/recall"], label="Recall")
axes[1,1].set_title("Recall")

# Empty slot
axes[1,2].axis("off")

# Styling
for ax in axes.flat:
    ax.set_xlabel("Epoch")
    ax.grid(True)

plt.tight_layout()
plt.show()

## Stage 5 — Model Validation

This stage evaluates the trained RF-DETR model on the configured
evaluation split.

### Tasks

- Load the trained checkpoint
- Run validation
- Collect detection metrics

In [ ]:
import os
from pycocotools.coco import COCO
from rfdetr import RFDETR

# Load checkpoint

checkpoint = os.path.join(
    cfg.result_dir,
    "checkpoint_best_regular.pth",
)

model = RFDETR.from_checkpoint(checkpoint)


/usr/local/lib/python3.12/dist-packages/deprecate/proxy.py:168: FutureWarning: The `RFDETRBase` was deprecated since v1.7.0. It will be removed in v2.0.0.
  stream(msg)
/usr/local/lib/python3.12/dist-packages/deprecate/proxy.py:168: FutureWarning: The `RFDETRLargeDeprecated` was deprecated since v1.7.0. It will be removed in v2.0.0.
  stream(msg)
/usr/local/lib/python3.12/dist-packages/deprecate/proxy.py:168: FutureWarning: The `RFDETRSegPreview` was deprecated since v1.7.0. It will be removed in v2.0.0.
  stream(msg)
[2026-07-25 09:20:50] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-25 09:20:50] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-25 09:20:51] [WARNING] rf-detr - load_pretrain_w

In [ ]:
import os
import time
import pandas as pd
import torch
from thop import profile

# CONFIG
summary_path = os.path.join(cfg.result_dir, "model_summary.txt")
benchmark_path = os.path.join(cfg.result_dir, "benchmark.csv")

warmup = 20
iters = 100
use_half = torch.cuda.is_available()

# MODEL
net = model.model.model
device = torch.device(model.model.device)

net.to(device)
net.eval()

if use_half:
    net.half()

dtype = torch.float16 if use_half else torch.float32

# MODEL SUMMARY
params = sum(p.numel() for p in net.parameters())
imgsz = model.model.resolution
dummy = torch.zeros(
    1,
    3,
    imgsz,
    imgsz,
    device=device,
    dtype=dtype,
)

macs, _ = profile(
    net,
    inputs=(dummy,),
    verbose=False,
)

gflops = macs * 2 / 1e9
grads = sum(p.numel() for p in net.parameters() if p.requires_grad)
with open(summary_path, "w") as f:
    f.write(f"{params:,} parameters\n")
    f.write(f"{grads:,} gradients\n")
    f.write(f"{gflops:.2f} GFLOPs\n")

print(open(summary_path).read())

# BENCHMARK
x = torch.zeros(
    1,
    3,
    imgsz,
    imgsz,
    device=device,
    dtype=dtype,
)

with torch.inference_mode():

    # Warmup
    for _ in range(warmup):
        _ = net(x)

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    for _ in range(iters):
        _ = net(x)

    if device.type == "cuda":
        torch.cuda.synchronize()

elapsed = time.perf_counter() - start

latency = elapsed * 1000 / iters
fps = 1000 / latency

benchmark = pd.DataFrame([
    {
        "Model": cfg.model_name,
        "Experiment": cfg.experiment_name,
        "Latency (ms)": round(latency, 3),
        "FPS": round(fps, 2),
        "GPU": (
            torch.cuda.get_device_name(device)
            if device.type == "cuda"
            else str(device).upper()
        ),
    }
])

benchmark.to_csv(
    benchmark_path,
    index=False,
)

print(benchmark)

In [ ]:
# Load test images
gt_path = os.path.join(
    cfg.coco_dir,
    "test",
    "_annotations.coco.json",
)

coco_gt = COCO(gt_path)

test_images = [
    os.path.join(cfg.coco_dir, "test", img["file_name"])
    for img in coco_gt.dataset["images"]
]

print(f"Total test images: {len(test_images)}")
# Predict
predictions = model.predict(
    images=test_images,
    threshold=0.001,
    include_source_image=False,
)

print(f"Predictions: {len(predictions)} images")

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Total test images: 23


[2026-07-25 09:20:52] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.optimize_for_inference(dtype=torch.float16).


Predictions: 23 images


## Stage 6 — COCO Evaluation

This stage calculates COCO-compatible detection metrics for RF-DETR.

### Tasks

- Prepare ground-truth annotations
- Prepare predictions
- Run COCO evaluation
- Collect AP metrics

In [ ]:
# Prepare ground-truth annotations for COCO evaluation.
import json
import os

num_classes = len(coco_gt.dataset["categories"])

predictions_json = []
# Convert RF-DETR predictions to COCO Detection Results
for image_path, det in zip(test_images, predictions):

    file_name = os.path.basename(image_path)


    image_id = os.path.splitext(file_name)[0]

    for box, score, cls in zip(
        det.xyxy,
        det.confidence,
        det.class_id,
    ):

        # Skip background prediction
        if cls >= num_classes:
            continue

        x1, y1, x2, y2 = box

        predictions_json.append(
            {
                "image_id": image_id,
                "category_id": int(cls)+1,
                "bbox": [
                    float(x1),
                    float(y1),
                    float(x2 - x1),
                    float(y2 - y1),
                ],
                "score": float(score),
            }
        )
# Save predictions.json
output_dir = os.path.join(cfg.result_dir, "validation")
os.makedirs(output_dir, exist_ok=True)

pred_path = os.path.join(output_dir, "predictions.json")

with open(pred_path, "w") as f:
    json.dump(predictions_json, f)

print(f"Saved {len(predictions_json)} predictions")
print(pred_path)

# Debug
print(predictions_json[0])

Saved 6848 predictions
/content/drive/MyDrive/Oral_Disease_Detection/output/dental_opg_2024/rfdetr-medium/exp4_e100_b8/validation/predictions.json
{'image_id': '104_jpg.rf.05501644f89b63b3f9ba38f2c2436c75', 'category_id': 1, 'bbox': [445.8642883300781, 208.51783752441406, 46.88470458984375, 66.84321594238281], 'score': 0.9165111184120178}


In [ ]:
# Calculate COCO AP metrics.
!python scripts/coco_evaluate.py \
    --data {cfg.yaml_path} \
    --project {cfg.output_dir} \
    --name {cfg.experiment_name}

Using existing COCO GT: /content/drive/MyDrive/Oral_Disease_Detection/output/dental_opg_2024/rfdetr-medium/exp4_e100_b8/validation/test_coco_gt.json
[exp4_e100_b8] Predictions: 1177
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.07s).
Accumulating evaluation results...
DONE (t=0.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.425
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.747
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.451
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.425
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.700
 Average Recall     (AR) @[ IoU=0.50:0.95 